# Pydantic

Pydantic is the most used data parsing and validation library in python and it's a must-have when data comes from outside such as APIs or LLMs.

Pydantic leverages type hints to enforce run-time data rules and throws a ValidationError at when they're violated.

## Install

\> poetry add pydantic

## Usage

Without Pydantic you should execute each validation by yourself inside \_\_init\_\_

In [83]:
class User:
    def __init__(self, name: str, age: int, email: str | None = None):
        if not isinstance(name, str):
            raise TypeError("name must be a string")
        if not isinstance(age, int):
            raise TypeError("age must be an integer")
        if email is not None and not isinstance(email, str):
            raise TypeError("email must be a string or None")
        
        self.name = name
        self.age = age
        self.email = email

    def __repr__(self):
        return f"name={self.name}, age={self.age}, email={self.email}"

user = User(name="Alice", age=30) 
print(user)

name=Alice, age=30, email=None


With Pydantic

In [84]:
from pydantic import BaseModel

class User(BaseModel): # you have to inherit from BaseModel
    name: str
    age: int
    email: str | None = None  # Optional field with default value None

user = User(name="Alice", age="30")  # age will be coerced to int automatically by pydantic so no exception will be thrown
print(user)

name='Alice' age=30 email=None


Serialize and convert to Json

In [82]:
print(user.model_dump()) # Convert to dictionary (useful for serialization)
print(user.model_dump_json()) # Convert to JSON string (useful to send data to APIs over the network)

{'name': 'Alice', 'age': 30, 'email': None}
{"name":"Alice","age":30,"email":null}


Read from API


In [69]:
data_from_api = {"name": "Bob", "age": 30, "email": None}

# model_validate is used to deserialize and validate a dictionary / json into a User instance
user = User.model_validate(data_from_api)
print(user)

# ** is the same as model_validate
user = User(**data_from_api)
print(user)

# model_validate is usually preferred because it's more flexible since it expose some parameters, like 'strict' that forbids coercion

name='Bob' age=30 email=None
name='Bob' age=30 email=None


Model_Config

In [42]:
from pydantic import ConfigDict

class StrictUser(BaseModel):
    model_config = ConfigDict(strict=True) # ConfigDict lets you config some pydantic parameters for the specific model
    name: str
    age: int

# strict forbid coercion
user = StrictUser(name = "George", age = 30) # OK
# user = StrictUser(name = "George", age = "30") # Validation Error since coercion is forbidden by strict=True

Fields

In [41]:
from pydantic import Field

class AdvancedUser(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    age: int = Field (gt=0, le=120, default= 30) # greater than, less or equal than, default value
    email: str | None = None
    website: str | None = Field(pattern=r"^https?://.*", description= "the user's website", default = None) # description is documentation, also used by LLMs

user = AdvancedUser(name = "George", age = 30)

Custom Field Validators

In [117]:
from pydantic import field_validator

class CustomUser(BaseModel):
    username: str

    @field_validator("username") # this has to match the name of the attribute it refers to
    @classmethod # this is a class method because we don't have an instance yet, we are validating the value before creating the instance
    def validate_username(cls, v): # v is the value being validated, cls is the class (not the instance because there is no instance yet)
        if " " in v:
            raise ValueError("username cannot contain spaces")
        return v.lower() # always return the value, here we normalize to lowercase

user = CustomUser(username="Alice")
# user = CustomUser(username="Alice A") not valid!
print(user)

username='alice'


Custom Model Validators

In [66]:
from pydantic import model_validator

class AnotherCustomUser(BaseModel):
    username: str
    password: str | None = None

    @model_validator(mode="before")
    @classmethod
    def check_no_password(cls, data: any) -> any:
        if isinstance(data, dict):
            if 'password' in data:
                raise ValueError("Password should not be provided")
        return data

    @model_validator(mode="after")
    def check_username_length(self) -> AnotherCustomUser:
        if len(self.username) < 3:
            raise ValueError("Username must be at least 3 characters long")
        return self
    
user = AnotherCustomUser(username="Alice")
print(user)

username='Alice' password=None


Nested Models

In [118]:
class OrderItem(BaseModel):
    product_id: int
    quantity: int = Field(gt=0, description="the quantity of the product ordered")

class Order(BaseModel):
    order_id: int
    items: list[OrderItem] = Field(min_length=1, description="the list of items in the order")

order = Order(order_id=1, items=[
    OrderItem(product_id=101, quantity=2), 
    OrderItem(product_id=102, quantity=1)
])

Generate Schema for documentation

In [50]:
class NewUser(BaseModel):
    name: str = Field(min_length=1, max_length=100, description="the user's name")
    age: int = Field(gt=0, le=120, default=30, description="the user's age")
    email: str | None = None

NewUser.model_json_schema()
"""
{
  "title": "NewUser",
  "type": "object",
  "properties": {
    "name": {
      "title": "Name",
      "type": "string",
      "description": "the user's name",
      "minLength": 1,
      "maxLength": 100
    },
    "age": {
      "title": "Age",
      "type": "integer",
      "description": "the user's age",
      "exclusiveMinimum": 0,
      "maximum": 120,
      "default": 30
    },
    "email": {
      "title": "Email",
      "anyOf": [
        {"type": "string"},
        {"type": "null"}
      ],
      "default": null
    }
  },
  "required": ["name"]
}
"""

'\n{\n  "title": "NewUser",\n  "type": "object",\n  "properties": {\n    "name": {\n      "title": "Name",\n      "type": "string",\n      "description": "the user\'s name",\n      "minLength": 1,\n      "maxLength": 100\n    },\n    "age": {\n      "title": "Age",\n      "type": "integer",\n      "description": "the user\'s age",\n      "exclusiveMinimum": 0,\n      "maximum": 120,\n      "default": 30\n    },\n    "email": {\n      "title": "Email",\n      "anyOf": [\n        {"type": "string"},\n        {"type": "null"}\n      ],\n      "default": null\n    }\n  },\n  "required": ["name"]\n}\n'

# Pydantic Settings

Pydantic can also be used to parse environment settings through a config class

## Install

\> poetry add pydantic-settings

## Usage

Retrieve from environment variables

In [127]:

import os
from pydantic_settings import BaseSettings

os.environ["API_KEY"] = "your_api_key_here"

class Settings(BaseSettings):
    api_key: str

# Reads from environment variables automatically and validates the values
# Field names are converted to uppercase and underscores, so api_key becomes API_KEY
print(Settings().api_key)

your_api_key_here


Retrieve from env file

In [ ]:
#create a .env file
#MY_APP_API_KEY="your_api_key_here"
#MY_APP_MAX_CONNECTIONS=10
#MY_APP_DEBUG="true"

from typing import Literal
from pydantic_settings import SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_prefix="MY_APP_")
    api_key: str
    max_connections: int = 100
    debug: bool = False
    environment: Literal["development", "staging", "production"] = "development"
    
    @property
    def is_production(self) -> bool:
        return self.environment == "production"

settings = Settings()
print(settings.api_key)
print(settings.max_connections)
print(settings.debug)
print(settings.environment)

your_api_key_here
10
True
development


Secrets

In [132]:
from pydantic import SecretStr
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    api_key: SecretStr

settings = Settings()

print(settings.api_key) # Secrets are hidden when printed # **********
actual_key = settings.api_key.get_secret_value() # Access the actual value when needed
print(actual_key)  # Oops! I accidentally revealed the secret key.

**********
your_api_key_here


Nested settings

In [140]:
os.environ["DATABASE__PORT"] = "1234"

class DatabaseSettings(BaseModel):
    host: str = "localhost"
    port: int = 5432
    name: str = "mydb"

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_nested_delimiter="__")   
     
    debug: bool = False
    database: DatabaseSettings = DatabaseSettings()

# Set via: DATABASE__HOST, DATABASE__PORT, DATABASE__NAME
Settings().model_dump()

{'debug': True,
 'database': {'host': 'localhost', 'port': 1234, 'name': 'mydb'}}

## Sources:
* https://pydantic.dev/docs/validation/latest/get-started/
* https://www.youtube.com/watch?v=PkQIREapb9o
* https://github.com/daveebbelaar/pydantic-crash-course
* https://github.com/Coding-Crashkurse/Pydantic-v2-crashcourse/blob/main/code.ipynb